In [97]:
import pandas as pd
import re

In [178]:
trace_file_1 = "../benchmarking/data/trace-20260925-55780271.txt"
trace_file_2 = "../benchmarking/data/trace-20260927-36157097.txt"
trace_file_3 = "../benchmarking/data/trace-20260928-46628641.txt"

In [179]:
trace_df = pd.read_csv(trace_file_1, delimiter="\t")
trace_extra = pd.read_csv(trace_file_2, delimiter="\t")
trace_ntsm = pd.read_csv(trace_file_3, delimiter="\t")

In [180]:
trace_extra = trace_extra[trace_extra["name"] == "CROSSCHECK_FINGERPRINTS"]
trace_df = trace_df[trace_df["status"] != "FAILED"]

In [181]:
trace_df = pd.concat([trace_df, trace_extra, trace_ntsm])

In [182]:
# Convert time to seconds
trace_df["realtime"] = pd.to_timedelta(trace_df["realtime"]).dt.total_seconds()
trace_df["duration"] = pd.to_timedelta(trace_df["duration"]).dt.total_seconds()

In [183]:
def convert_size_to_gb(val):
    if pd.isna(val):
        return None
    
    # Clean and split string into numeric part and unit part
    val_str = str(val).strip().lower()
    match = re.match(r'^([\d.]+)\s*([a-z]+)$', val_str)
    
    if not match:
        return float('nan')
    
    num, unit = float(match.group(1)), match.group(2)
    
    # Define multipliers to convert everything to GB
    if unit in ['MB', 'mb', 'm']:
        return num / 1024  # Convert MB to GB
    elif unit in ['GB', 'gb', 'g']:
        return num
    elif unit in ['KB', 'kb', 'k']:
        return num / (1024 ** 2)  # Convert KB to GB
    
    return num

In [184]:
trace_df[trace_df["name"].str.contains("GENOTYPE_INDIVIDUAL")]

,task_id,hash,native_id,name,status,exit,submit,duration,realtime,%cpu,peak_rss,peak_vmem,rchar,wchar
53,146,b1/5882e3,32981153,GENOTYPE_INDIVIDUAL (HGSOC-2507_ds),COMPLETED,0,2026-09-25 15:31:18.150,580.0,570.0,99.7%,502.1 MB,7.1 GB,8.7 GB,846 MB
114,153,c3/998d61,32981397,GENOTYPE_INDIVIDUAL (HGSOC-2416_ds),COMPLETED,0,2026-09-25 15:33:03.113,910.0,862.0,101.3%,500.2 MB,7.1 GB,8.6 GB,925.5 MB
144,148,6f/776133,32982026,GENOTYPE_INDIVIDUAL (HGSOC-2526_ds),COMPLETED,0,2026-09-25 15:48:13.184,205.0,163.0,94.9%,497.2 MB,7.1 GB,4.7 GB,495.4 MB
166,157,ec/15ebb0,32981569,GENOTYPE_INDIVIDUAL (HGSOC-2186_ds),COMPLETED,0,2026-09-25 15:40:58.189,815.0,808.0,101.5%,505.4 MB,7.1 GB,6.8 GB,756.9 MB
205,189,c8/fb2b04,32982230,GENOTYPE_INDIVIDUAL (HGSOC-2477_ds),COMPLETED,0,2026-09-25 15:54:33.432,455.0,422.0,101.9%,503.1 MB,7.1 GB,9 GB,831.1 MB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1421,230,03/6516be,32989416,GENOTYPE_INDIVIDUAL (HGSOC-2408_ds),COMPLETED,0,2026-09-25 20:13:19.725,299.0,289.0,103.4%,499.4 MB,7.1 GB,6.2 GB,595.4 MB
1422,236,e1/9b9700,32989502,GENOTYPE_INDIVIDUAL (HGSOC-2246_ds),COMPLETED,0,2026-09-25 20:15:13.712,585.0,561.0,102.7%,508.3 MB,7.1 GB,8.6 GB,907.2 MB
1423,254,c8/2b5f2a,32989846,GENOTYPE_INDIVIDUAL (HGSOC-2466_ds),COMPLETED,0,2026-09-25 20:24:58.735,155.0,143.0,106.8%,494.8 MB,7.1 GB,4.4 GB,470.7 MB
1424,252,03/5bcbfa,32989601,GENOTYPE_INDIVIDUAL (HGSOC-2221_ds),COMPLETED,0,2026-09-25 20:18:18.720,695.0,685.0,102.5%,500.7 MB,7.1 GB,10.1 GB,964 MB


In [187]:
# Analyze the computational burden for a single process
# Let's start with PREPARE_INPUTS
# We'll build up a df that has a different process in each row and columns for various metrics like realtime and peak_rss
# Columns will be: process, n (number of occurrences), mean realtime (total time in seconds), mean peak_rss (maximum memory usage in MB), mean %cpu, mean rchar in GB, mean wchar in GB
processes = [
    "PREPARE_INPUTS", "GENOTYPE_INDIVIDUAL", "MERGE", "FILTER_BAM",
    "SNP_REF", "CROSSCHECK", "HYSYS", "NGSCHECKMATE", "PEDDY", "TIMEATTACK",
    "OMICSPRINT", "CONPAIR", "SOMALIER_EXTRACT", "SOMALIER_RELATE", 
    "NTSM_COUNT", "NTSM_EVAL", "VIREO"
    ] 
metrics_df = pd.DataFrame()

for process in processes:
    process_df = trace_df[trace_df["name"].str.contains(process)]
    n = len(process_df)
    mean_realtime = process_df["realtime"].mean()
    mean_peak_rss = process_df["peak_rss"].apply(convert_size_to_gb).astype(float).mean()  # Convert to GB
    mean_cpu = process_df["%cpu"].str.rstrip("%").astype(float).mean()
    mean_rchar = process_df["rchar"].apply(convert_size_to_gb).astype(float).mean()  # Convert to GB
    mean_wchar = process_df["wchar"].apply(convert_size_to_gb).astype(float).mean()  # Convert to GB
    metrics_df = pd.concat([
        metrics_df, 
        pd.DataFrame([{
        "process": process,
        "n": n,
        "mean realtime (s)": mean_realtime,
        "mean peak_rss (GB)": mean_peak_rss,
        "mean %cpu": mean_cpu,
        "mean rchar (GB)": mean_rchar,
        "mean wchar (GB)": mean_wchar
    }])])

metrics_df



,process,n,mean realtime (s),mean peak_rss (GB),mean %cpu,mean rchar (GB),mean wchar (GB)
0,PREPARE_INPUTS,68,23.873294,0.027008,92.770588,3.467172,0.003266
0,GENOTYPE_INDIVIDUAL,68,391.691176,0.490523,102.114706,7.130882,0.716375
0,MERGE,1,571.000000,0.626465,56.500000,2.900000,0.643848
0,FILTER_BAM,68,313.394441,0.022952,95.744118,6.070817,2.606835
0,SNP_REF,1,41.600000,0.692773,70.700000,2.200000,1.900000
0,CROSSCHECK,1,67.000000,0.546191,138.900000,1.800000,1.500000
0,HYSYS,1,441.000000,1.300000,50.500000,2.800000,2.600000
0,NGSCHECKMATE,1,222.000000,0.149414,8.600000,1.400000,1.100000
0,PEDDY,1,84.000000,0.662988,9.700000,0.912500,0.004004
0,TIMEATTACK,0,NaN,NaN,NaN,NaN,NaN


In [186]:
metrics_df.to_csv("computational_burden_averages_hgsoc.csv", index=False)

In [83]:
trace_df[trace_df["name"].str.contains("GENOTYPE_INDIVIDUAL")]["peak_rss"].str.rstrip(" MB").astype(float).describe()

count     68.000000
mean     502.295588
std        4.789959
min      494.100000
25%      499.900000
50%      501.000000
75%      503.850000
max      523.600000
Name: peak_rss, dtype: float64